# GPT-3.5 Chain-of-Thought Evaluation

This notebook evaluates GPT-3.5-turbo on poker strategy tasks using chain-of-thought reasoning.

## Setup

In [1]:
import os
import json
import random
import time
from tqdm import tqdm
from dotenv import load_dotenv
load_dotenv()

try:
    from openai import OpenAI
except Exception as _:
    import openai as _openai
    OpenAI = getattr(_openai, "OpenAI", None) or getattr(_openai, "OpenAI", None)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY environment variable before running this notebook")

client = OpenAI(api_key=OPENAI_API_KEY)
model = "gpt-3.5-turbo"

# temperatures to evaluate
temps = [0.1, 0.3, 0.5, 0.7, 1.0]

print("Using model:", model)

Using model: gpt-3.5-turbo


Import dependencies and configure OpenAI client

In [ ]:
# base_dir = 'llm'
file_no_ans = 'datasets/pokerbench_cot_no_answer.json'
file_with_ans = 'datasets/pokerbench_cot_with_answer.json'
file_direct = 'datasets/pokerbench_direct_answer.json'

with open(file_no_ans, 'r', encoding='utf-8') as f:
    cot_no = json.load(f)

with open(file_with_ans, 'r', encoding='utf-8') as f:
    cot_with = json.load(f)

with open(file_direct, 'r', encoding='utf-8') as f:
    direct = json.load(f)

if len(cot_no) != len(cot_with):
    print(f'Warning: lengths differ (no_answer={len(cot_no)}, with_answer={len(cot_with)})')

# Create dataset for evaluation
every_n_prompts = 20
sample_idx = list(range(0, len(cot_no), every_n_prompts))
print(f'Selected {len(sample_idx)} examples (out of {len(cot_no)})')

Selected 230 examples (out of 4600)


Load datasets and create evaluation sample

## API Execution

Define async functions for concurrent API requests

In [3]:
import asyncio

output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

# Rate limiter: 0.4s delay = 2.5 req/sec (conservative for API limits)
RATE_LIMIT_DELAY = 0.40

system_prompt = """
You are an expert in 6-max No-Limit Hold'em strategy. Your job is to analyze a single poker hand at a time and produce a logically correct explanation of the optimal action.

Follow these rules:
(1) Use only the information explicitly provided. If something is not stated, mark it as unknown and do not fabricate details.
(2) Evaluate the spot using position ranges, board texture, nut advantage, range interaction by street, equity distribution, pot odds, SPR, and the opponent's assumed line (polarized/merged/capped).
(3) If you make any assumptions (such as typical BB defend ranges), state them explicitly and justify why they are reasonable.
(4) Break down the logic step-by-step and ensure internal consistency. Point out any uncertainties or possible logical failure points.
(5) Prioritize correctness over confidence. If multiple actions are close in EV, say so and explain when each would be preferred.
(6) Use "\\n" for new lines and use card emojis for suits (♠️♥️♦️♣️).
(7) Make sure all responses end in the exact format: "FINAL DECISION ::: <FOLD / CHECK / CALL / BET X / RAISE X / ALL IN>"
(8) Limit the response to a maximum of 180 words.
"""

async def call_api(idx, temp, prompt_text, ref_answer, semaphore):
    """Make API call with semaphore to limit concurrent requests."""
    async with semaphore:
        try:
            # Use sync client in thread pool to avoid blocking
            resp = await asyncio.to_thread(
                client.chat.completions.create,
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt_text}
                ],
                temperature=temp,
            )
            text = None
            try:
                text = resp.choices[0].message.content.strip()
            except Exception:
                text = getattr(resp.choices[0], 'text', None) or str(resp)
            usage = getattr(resp, 'usage', None)
        except Exception as e:
            print(f'Request error at idx {idx}, temp {temp}:', repr(e))
            text = None
            usage = None
        
        return {
            'index': idx,
            'temperature': temp,
            'prompt': prompt_text,
            'reference': ref_answer,
            'response': text,
            'usage': str(usage) if usage else None,
        }

async def process_temperature(temp, output_file, prompts, references, max_concurrent=25):
    """Process all prompts for a given temperature with batching."""
    semaphore = asyncio.Semaphore(max_concurrent)
    tasks = [call_api(idx, temp, prompts[idx], references[idx], semaphore) for idx in sample_idx]
    
    results = []
    for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc=f'temp {temp}'):
        result = await coro
        results.append(result)
    
    # Sort by index before writing (to maintain order)
    results.sort(key=lambda x: x['index'])
    with open(output_file, 'w', encoding='utf-8') as f:
        for record in results:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

async def main():
    """Run evaluation for all temperatures and both datasets (cot_no and cot_with)."""
    
    # Run cot_no (without answer)
    print("=" * 60)
    print("Running CoT WITHOUT answer dataset (cot_no)")
    print("=" * 60)
    for temp in temps:
        out_path = os.path.join(output_dir, f'{model}_cot_no_temp{str(temp).replace(".", "")}.jsonl')
        print(f'\nWriting outputs to {out_path}')
        await process_temperature(temp, out_path, cot_no, cot_with, max_concurrent=8)
        print(f'Finished temperature {temp}')
    
    # Run cot_with (with answer)
    print("\n" + "=" * 60)
    print("Running CoT WITH answer dataset (cot_with)")
    print("=" * 60)
    for temp in temps:
        out_path = os.path.join(output_dir, f'{model}_cot_with_temp{str(temp).replace(".", "")}.jsonl')
        print(f'\nWriting outputs to {out_path}')
        await process_temperature(temp, out_path, cot_with, cot_no, max_concurrent=8)
        print(f'Finished temperature {temp}')

# Run the async main function
await main()
print('\nAll temperatures complete for both datasets. Files are in the results/ directory.')


Running CoT WITHOUT answer dataset (cot_no)

Writing outputs to results\gpt-3.5-turbo_cot_no_temp01.jsonl


temp 0.1: 100%|██████████| 230/230 [00:59<00:00,  3.84it/s]


Finished temperature 0.1

Writing outputs to results\gpt-3.5-turbo_cot_no_temp03.jsonl


temp 0.3: 100%|██████████| 230/230 [00:58<00:00,  3.94it/s]


Finished temperature 0.3

Writing outputs to results\gpt-3.5-turbo_cot_no_temp05.jsonl


...

Finished temperature 0.5

Writing outputs to results\gpt-3.5-turbo_cot_with_temp07.jsonl


temp 0.7: 100%|██████████| 230/230 [01:05<00:00,  3.49it/s]


Finished temperature 0.7

Writing outputs to results\gpt-3.5-turbo_cot_with_temp10.jsonl


temp 1.0: 100%|██████████| 230/230 [01:00<00:00,  3.77it/s]

Finished temperature 1.0

All temperatures complete for both datasets. Files are in the results/ directory.


Run baseline no-cot version

Run main evaluation loop for all temperatures

In [11]:
# run base

async def main_direct():
    """Run evaluation for direct answer dataset (no CoT)."""
    
    # Run direct (without CoT)
    print("=" * 60)
    print("Running DIRECT answer dataset (direct_out)")
    print("=" * 60)
    out_path = os.path.join(output_dir, f'{model}_direct.jsonl')
    print(f'\nWriting outputs to {out_path}')
    await process_temperature(0.0, out_path, direct, direct, max_concurrent=8)
    print(f'Finished DIRECT answer dataset')

# Run the async main function
await main_direct()
print('\nDIRECT answer dataset complete. File is in the results/ directory.')

Running DIRECT answer dataset (direct_out)

Writing outputs to results\gpt-3.5-turbo_direct.jsonl


temp 0.0: 100%|██████████| 230/230 [00:16<00:00, 14.08it/s]

Finished DIRECT answer dataset

DIRECT answer dataset complete. File is in the results/ directory.


## Get accuracy of CoT_no_answer

Write to csv

Run baseline evaluation without chain-of-thought

In [13]:
import re

# Load all cot_no results across all temperatures
all_results = {}
for temp in temps:
    result_file = os.path.join(output_dir, f'{model}_cot_no_temp{str(temp).replace(".", "")}.jsonl')
    if os.path.exists(result_file):
        with open(result_file, 'r', encoding='utf-8') as f:
            results = [json.loads(line) for line in f]
            all_results[temp] = {r['index']: r for r in results}

# Also load direct results
direct_result_file = os.path.join(output_dir, f'{model}_direct.jsonl')
if os.path.exists(direct_result_file):
    with open(direct_result_file, 'r', encoding='utf-8') as f:
        results = [json.loads(line) for line in f]
        all_results['direct'] = {r['index']: r for r in results}

# load ground truth references from balanced_4600
reference_file = "datasets/pokerbench_balanced_4600.json"
with open(reference_file, 'r', encoding='utf-8') as f:
    references = json.load(f)
ground_truth = []
for stage_vals in references.values():
    ground_truth.extend([inst["output"] for inst in stage_vals])


print(f"Loaded results for temperatures: {list(all_results.keys())}")

def extract_final_decision(response_text):
    """Extract FINAL DECISION from response text."""
    if not response_text:
        return None
    # Look for "FINAL DECISION :::" pattern
    match = re.search(r'FINAL DECISION\s*:::\s*(.+?)(?:\n|$)', response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip().lower()
    return None

def categorize_by_stage(idx):
    """Categorize index into poker stage based on range."""
    if idx < 600:
        return 'preflop'
    elif idx < 1600:
        return 'flop'
    elif idx < 3100:
        return 'turn'
    else:
        return 'river'

# Parse and organize results by stage and temperature
parsed_results = {temp: [] for temp in temps}
parsed_results['direct'] = []
for temp, results_dict in all_results.items():
    for idx, record in results_dict.items():
        decision = extract_final_decision(record['response'])
        stage = categorize_by_stage(idx)
        parsed_results[temp].append({
            'index': idx,
            'stage': stage,
            'decision': decision,
            'reference': ground_truth[idx] if idx < len(ground_truth) else None,
        })

# write to 5 seperate csv files by temperature
import csv
for temp in temps:
    csv_file = os.path.join(output_dir, f'parsed_results_temp{str(temp).replace(".", "")}.csv')
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['index', 'stage', 'decision', 'reference'])
        writer.writeheader()
        for record in parsed_results[temp]:
            writer.writerow(record)
    print(f'Wrote parsed results to {csv_file}')

# Also write direct results if available
if 'direct' in parsed_results and parsed_results['direct']:
    csv_file = os.path.join(output_dir, f'parsed_results_direct.csv')
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['index', 'stage', 'decision', 'reference'])
        writer.writeheader()
        for record in parsed_results['direct']:
            writer.writerow(record)
    print(f'Wrote parsed direct results to {csv_file}')


Loaded results for temperatures: [0.1, 0.3, 0.5, 0.7, 1.0, 'direct']
Wrote parsed results to results\parsed_results_temp01.csv
Wrote parsed results to results\parsed_results_temp03.csv
Wrote parsed results to results\parsed_results_temp05.csv
Wrote parsed results to results\parsed_results_temp07.csv
Wrote parsed results to results\parsed_results_temp10.csv
Wrote parsed direct results to results\parsed_results_direct.csv


## Results Processing + Evalulation

Parse API responses and categorize by poker stage
Evaluate predictions against ground truth and classify as correct/semi-correct/incorrect/invalid

In [ ]:
import csv

for temp in temps:
    csv_file = os.path.join(output_dir, f'parsed_results_temp{str(temp).replace(".", "")}.csv')
    print(f'\nSample results from {csv_file}:\n')
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        results = list(reader)

In [ ]:
import re

def parse_action(action_str):
    """
    Parse action string into (action_type, amount).
    Returns: (action, amount) where action is one of: fold, call, check, bet, raise, all-in
    """
    if not action_str or action_str == 'None':
        return (None, None)
    
    action_str = action_str.strip().lower()
    
    # Check for all-in
    if 'all in' in action_str or 'all-in' in action_str:
        return ('all-in', None)
    
    # Check for simple actions
    if 'fold' in action_str:
        return ('fold', None)
    if 'call' in action_str:
        return ('call', None)
    if 'check' in action_str:
        return ('check', None)
    
    # Check for bet/raise with amount
    bet_match = re.search(r'bet\s+(\d+(?:\.\d+)?)', action_str)
    if bet_match:
        return ('bet', float(bet_match.group(1)))
    
    raise_match = re.search(r'raise\s+(\d+(?:\.\d+)?)', action_str)
    if raise_match:
        return ('raise', float(raise_match.group(1)))
    
    # If we can't parse, return None
    return (None, None)

def is_action_valid(predicted_action, predicted_amount, reference_action, reference_amount):
    """
    Check if predicted action makes sense given reference action context.
    Returns: True if valid, False if invalid
    
    Invalid means:
    - check instead of call or vice versa
    - fold instead of bet/check or vice versa
    - bet instead of raise or vice versa
    """
    if predicted_action is None:
        return False
    
    # Check invalid swaps
    if predicted_action == 'check' and reference_action in {'call', 'raise'}:
        return False
    if predicted_action in {'call', 'raise'} and reference_action == 'check':
        return False
    
    # folding with no money in pot
    if predicted_action == 'fold' and reference_action in {'bet', 'check'}:
        return False
    if predicted_action in {'bet', 'check'} and reference_action == 'fold':
        return False
    
    # bet vs raise confusion
    if predicted_action == 'bet' and reference_action == 'raise':
        return False
    if predicted_action == 'raise' and reference_action == 'bet':
        return False
    
    return True

def evaluate_prediction(predicted, reference):
    """
    Evaluate a single prediction against reference.
    Returns: 'correct', 'semi-correct', 'incorrect', or 'invalid'
    """
    pred_action, pred_amount = parse_action(predicted)
    ref_action, ref_amount = parse_action(reference)
    
    # Check if we could parse the prediction
    if pred_action is None:
        return 'invalid'
    
    # Check if action is contextually invalid
    if not is_action_valid(pred_action, pred_amount, ref_action, ref_amount):
        return 'invalid'
    
    # If actions don't match at all, it's incorrect
    if pred_action != ref_action:
        # Special case: all-in can match raise with large amount
        if pred_action == 'all-in' and ref_action in {'bet','raise'}:
            if ref_amount > 50:
                return 'correct'
            else:
                return 'semi-correct'
        return 'incorrect'
    
    # Actions match - check amounts if applicable
    if pred_action in {'bet', 'raise'}:
        if pred_amount is None or ref_amount is None:
            return 'invalid'
        
        # Calculate if within 50% range
        lower_bound = ref_amount * 0.5
        upper_bound = ref_amount * 1.5
        
        if lower_bound <= pred_amount <= upper_bound:
            return 'correct'
        else:
            return 'semi-correct'
    
    # For fold, call, check, all-in - if action matches, it's correct
    return 'correct'

# Load all CSV files and evaluate
results_by_temp = {}

all_temps = list(temps) + ['direct']

for temp in all_temps:
    if temp == 'direct':
        csv_file = os.path.join(output_dir, 'parsed_results_direct.csv')
    else:
        csv_file = os.path.join(output_dir, f'parsed_results_temp{str(temp).replace(".", "")}.csv')
    
    if not os.path.exists(csv_file):
        print(f"Skipping {csv_file} - file not found")
        continue
    
    with open(csv_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    
    # Evaluate each row
    evaluations = []
    for row in rows:
        result = evaluate_prediction(row['decision'], row['reference'])
        evaluations.append({
            'index': int(row['index']),
            'stage': row['stage'],
            'predicted': row['decision'],
            'reference': row['reference'],
            'result': result
        })
    
    results_by_temp[temp] = evaluations

    # overwrite csv with results
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['index', 'stage', 'decision', 'reference', 'result'])
        writer.writeheader()
        for ev in evaluations:
            writer.writerow({
                'index': ev['index'],
                'stage': ev['stage'],
                'decision': ev['predicted'],
                'reference': ev['reference'],
                'result': ev['result']
            })



ACCURACY STATISTICS BY TEMPERATURE

Temperature: 0.1
  Total:          230
  Correct:        100 ( 43.5%)
  Semi-correct:     2 (  0.9%)
  Incorrect:      112 ( 48.7%)
  Invalid:         16 (  7.0%)

Temperature: 0.3
  Total:          230
  Correct:        112 ( 48.7%)
  Semi-correct:     0 (  0.0%)
  Incorrect:      104 ( 45.2%)
  Invalid:         14 (  6.1%)

Temperature: 0.5
  Total:          230
  Correct:         98 ( 42.6%)
  Semi-correct:     1 (  0.4%)
  Incorrect:      115 ( 50.0%)
  Invalid:         16 (  7.0%)

Temperature: 0.7
  Total:          230
  Correct:        104 ( 45.2%)
  Semi-correct:     2 (  0.9%)
  Incorrect:      107 ( 46.5%)
  Invalid:         17 (  7.4%)

Temperature: 1.0
  Total:          230
  Correct:        107 ( 46.5%)
  Semi-correct:     0 (  0.0%)
  Incorrect:      105 ( 45.7%)
  Invalid:         18 (  7.8%)

Temperature: direct
  Total:          230
  Correct:        125 ( 54.3%)
  Semi-correct:     0 (  0.0%)
  Incorrect:       97 ( 42.2%)
  Invalid

## Report Generation

Generate comprehensive accuracy report across temperatures and poker stages

In [16]:
# Generate comprehensive accuracy report
from collections import Counter

report_lines = []

def add_line(line=""):
    report_lines.append(line)

# Header
add_line("=" * 80)
add_line("GPT-3.5-TURBO POKER STRATEGY EVALUATION REPORT")
add_line("=" * 80)
add_line()
add_line(f"Generated: {time.strftime('%Y-%m-%d %H:%M:%S')}")
add_line(f"Model: {model}")
add_line(f"Dataset: PokerBench (230 examples sampled every 20th prompt)")
add_line()

# Overall accuracy by temperature
add_line("=" * 80)
add_line("1. OVERALL ACCURACY BY TEMPERATURE")
add_line("=" * 80)
add_line()

for temp in all_temps:
    if temp not in results_by_temp:
        continue
    
    evals = results_by_temp[temp]
    total = len(evals)
    
    correct = sum(1 for e in evals if e['result'] == 'correct')
    semi_correct = sum(1 for e in evals if e['result'] == 'semi-correct')
    incorrect = sum(1 for e in evals if e['result'] == 'incorrect')
    invalid = sum(1 for e in evals if e['result'] == 'invalid')
    
    add_line(f"Temperature: {temp}")
    add_line(f"  Total:         {total:4}")
    add_line(f"  Correct:       {correct:4} ({correct/total*100:5.1f}%)")
    add_line(f"  Semi-correct:  {semi_correct:4} ({semi_correct/total*100:5.1f}%)")
    add_line(f"  Incorrect:     {incorrect:4} ({incorrect/total*100:5.1f}%)")
    add_line(f"  Invalid:       {invalid:4} ({invalid/total*100:5.1f}%)")
    add_line()

# Accuracy by stage for each temperature
add_line("=" * 80)
add_line("2. ACCURACY BY STAGE (ALL TEMPERATURES)")
add_line("=" * 80)
add_line()

for temp in all_temps:
    if temp not in results_by_temp:
        continue
    
    add_line(f"Temperature: {temp}")
    add_line("-" * 60)
    
    for stage in ['preflop', 'flop', 'turn', 'river']:
        stage_evals = [e for e in results_by_temp[temp] if e['stage'] == stage]
        total = len(stage_evals)
        
        if total == 0:
            continue
        
        correct = sum(1 for e in stage_evals if e['result'] == 'correct')
        semi_correct = sum(1 for e in stage_evals if e['result'] == 'semi-correct')
        incorrect = sum(1 for e in stage_evals if e['result'] == 'incorrect')
        invalid = sum(1 for e in stage_evals if e['result'] == 'invalid')
        
        add_line(f"  {stage.upper():8} - Correct: {correct:3}/{total:3} ({correct/total*100:5.1f}%) | "
                f"Semi: {semi_correct:3} ({semi_correct/total*100:5.1f}%) | "
                f"Wrong: {incorrect:3} ({incorrect/total*100:5.1f}%) | "
                f"Invalid: {invalid:3} ({invalid/total*100:5.1f}%)")
    add_line()

# Action distribution analysis
add_line("=" * 80)
add_line("3. ACTION DISTRIBUTION ANALYSIS")
add_line("=" * 80)
add_line()

for temp in all_temps:
    if temp not in results_by_temp:
        continue
    
    evals = results_by_temp[temp]
    
    # Extract action types from predictions and references
    pred_actions = []
    ref_actions = []
    
    for ev in evals:
        pred_act, _ = parse_action(ev['predicted'])
        ref_act, _ = parse_action(ev['reference'])
        if pred_act:
            pred_actions.append(pred_act)
        if ref_act:
            ref_actions.append(ref_act)
    
    pred_counts = Counter(pred_actions)
    ref_counts = Counter(ref_actions)
    
    add_line(f"Temperature: {temp}")
    add_line("-" * 60)
    add_line(f"{'Action':<10} {'Predicted':<12} {'Reference':<12} {'Diff':<10}")
    add_line("-" * 60)
    
    all_actions = set(pred_counts.keys()) | set(ref_counts.keys())
    for action in sorted(all_actions):
        pred_count = pred_counts.get(action, 0)
        ref_count = ref_counts.get(action, 0)
        diff = pred_count - ref_count
        diff_str = f"{diff:+d}" if diff != 0 else "0"
        add_line(f"{action:<10} {pred_count:<12} {ref_count:<12} {diff_str:<10}")
    add_line()

# Error pattern analysis
add_line("=" * 80)
add_line("4. ERROR PATTERN ANALYSIS")
add_line("=" * 80)
add_line()

for temp in all_temps:
    if temp not in results_by_temp:
        continue
    
    evals = results_by_temp[temp]
    
    # Analyze incorrect predictions
    incorrect_evals = [e for e in evals if e['result'] in ['incorrect', 'invalid']]
    
    if not incorrect_evals:
        continue
    
    add_line(f"Temperature: {temp}")
    add_line("-" * 60)
    
    # Count error types
    error_patterns = []
    for ev in incorrect_evals:
        pred_act, _ = parse_action(ev['predicted'])
        ref_act, _ = parse_action(ev['reference'])
        if pred_act and ref_act:
            error_patterns.append(f"{ref_act} → {pred_act}")
    
    pattern_counts = Counter(error_patterns)
    add_line(f"Most common errors (Reference → Predicted):")
    for pattern, count in pattern_counts.most_common(10):
        add_line(f"  {pattern:<20} {count:3} times")
    add_line()

# Stage-specific insights
add_line("=" * 80)
add_line("5. STAGE-SPECIFIC INSIGHTS (Temperature 0.1)")
add_line("=" * 80)
add_line()

if 0.1 in results_by_temp:
    for stage in ['preflop', 'flop', 'turn', 'river']:
        stage_evals = [e for e in results_by_temp[0.1] if e['stage'] == stage]
        
        if not stage_evals:
            continue
        
        add_line(f"{stage.upper()}")
        add_line("-" * 60)
        
        total = len(stage_evals)
        correct = sum(1 for e in stage_evals if e['result'] == 'correct')
        
        # Action distributions
        pred_actions = []
        ref_actions = []
        for ev in stage_evals:
            pred_act, _ = parse_action(ev['predicted'])
            ref_act, _ = parse_action(ev['reference'])
            if pred_act:
                pred_actions.append(pred_act)
            if ref_act:
                ref_actions.append(ref_act)
        
        pred_counts = Counter(pred_actions)
        ref_counts = Counter(ref_actions)
        
        add_line(f"Accuracy: {correct}/{total} ({correct/total*100:.1f}%)")
        add_line(f"Most common predicted actions: {', '.join(f'{a}({c})' for a, c in pred_counts.most_common(3))}")
        add_line(f"Most common reference actions: {', '.join(f'{a}({c})' for a, c in ref_counts.most_common(3))}")
        add_line()

# Summary and conclusions
add_line("=" * 80)
add_line("6. SUMMARY")
add_line("=" * 80)
add_line()

best_temp = None
best_accuracy = 0

for temp in all_temps:
    if temp not in results_by_temp:
        continue
    
    evals = results_by_temp[temp]
    correct = sum(1 for e in evals if e['result'] == 'correct')
    accuracy = correct / len(evals) if evals else 0
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_temp = temp

add_line(f"Best performing temperature: {best_temp} ({best_accuracy*100:.1f}% accuracy)")
add_line()

# Calculate overall trends
if len(all_temps) > 1:
    add_line("Observations:")
    add_line(f"- Total evaluations: {len(results_by_temp[all_temps[0]]) if all_temps[0] in results_by_temp else 0}")
    add_line(f"- Poker stages covered: Preflop, Flop, Turn, River")
    add_line(f"- Evaluation metrics: Correct (exact or within 50% bet size), Semi-correct (right action, wrong size),")
    add_line(f"                      Incorrect (wrong action), Invalid (unparsable or contextually wrong)")

add_line()
add_line("=" * 80)
add_line("END OF REPORT")
add_line("=" * 80)

# Write report to file
report_path = os.path.join(output_dir, 'report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print(f"Report written to: {report_path}")
print(f"\nReport summary:")
print(f"  Best temperature: {best_temp} ({best_accuracy*100:.1f}% accuracy)")
print(f"  Report length: {len(report_lines)} lines")

Report written to: results\report.txt

Report summary:
  Best temperature: direct (54.3% accuracy)
  Report length: 301 lines
